In [1]:
import cv2


In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

# IMPORTANT: Sirf FOLDER path append karein, util.py mat likhein
sys.path.append('/home/harish/computer_vision/parking')

from util import empty_or_not, get_parking_spots_bboxes

def calc_diff(im1,im2):
    return np.abs(np.mean(im1)-np.mean(im2))

# File Paths
Video_path = '/home/harish/computer_vision/parking/parking_1920_1080_loop.mp4'
mask_crop_path = '/home/harish/computer_vision/parking/mask_1920_1080.png'

# Load mask image in grayscale
mask = cv2.imread(mask_crop_path, 0)

# Connected components analysis on mask image array
connected_components = cv2.connectedComponentsWithStats(mask, 4, cv2.CV_32S)

# Extract parking spots bboxes using util function
spots = get_parking_spots_bboxes(connected_components)
print("Total spots detected:", len(spots))

# Open Video Capture
cap = cv2.VideoCapture(Video_path)

# Status list initialize karein (naam: spots_status)
spots_status = [None for _ in spots]
diffs = [None for _ in spots]
previous_frame = None
step = 30
frame_nmr = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_nmr % step == 0 and previous_frame is not None:
            for spot_indx, spot in enumerate(spots):
                 x1, y1, w, h = spot
                 spot_crop = frame[y1 : y1 + h, x1 : x1 + w, :]
                 diffs[spot_indx]=calc_diff(spot_crop,previous_frame[y1 : y1 + h, x1 : x1 + w, :])
          # 1. Print ke liye list conversion
        #     print([diffs[j] for j in np.argsort(diffs)[::-1]])

        #  # 2. plt.hist ke liye list comprehension [...]
        #     max_diff = np.max(diffs) if np.max(diffs) > 0 else 1
        #     plt.figure()
        #     plt.hist([diffs[j] / max_diff for j in np.argsort(diffs)[::-1]])
        #     if frame_nmr == 300:
        #         plt.show()
    # Har 'step' frames baad ML model run karein (for optimization)
    if frame_nmr % step == 0:
        if previous_frame==None:
            arr_=range(len(spots))
        else:
            arr_=[j for j in np.argsort(diffs) if diffs[j]/np.amax(diffs)>0.4]    
        for spot_indx in arr_:
            x1, y1, w, h = spots[spot_indx]
            spot_crop = frame[y1 : y1 + h, x1 : x1 + w, :]

            # Individual status nikaal kar list me save karein
            st = empty_or_not(spot_crop)
            spots_status[spot_indx] = st
    if frame_nmr % step == 0:
        previous_frame=frame.copy()        

    # Parking bounding boxes draw karein
    for spot_indx, spot in enumerate(spots):
        x1, y1, w, h = spot
        status = spots_status[spot_indx]

        if status:
            # Green rectangle if spot is EMPTY
            cv2.rectangle(frame, (x1, y1), (x1 + w, y1 + h), (0, 255, 0), 2)
        else:
            # Red rectangle if spot is NOT EMPTY
            cv2.rectangle(frame, (x1, y1), (x1 + w, y1 + h), (0, 0, 255), 2)

    #cv2.putText(frame, "available_spots: {}/{}".format(str(sum(spots_status),str(len(spots_status)))),(100,60),cv2.FONT_HERSHEY_NORMAL,1,(255,255,255))        
    # ✅ Fixed Line:
    cv2.putText(
    frame,
    "Available Spots: {}/{}".format(sum(spots_status), len(spots_status)),
    (100, 60),
    cv2.FONT_HERSHEY_SIMPLEX,
    1,
    (255, 255, 255),
    2,
    )
    cv2.imshow("Parking Spot Detection", frame)

    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

    frame_nmr += 1

cap.release()
cv2.destroyAllWindows()

Total spots detected: 396


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

: 

In [1]:
import os
import sys
import cv2
import matplotlib.pyplot as plt
import numpy as np

# IMPORTANT: Sirf FOLDER path append karein, util.py mat likhein
sys.path.append("/home/harish/computer_vision/parking")

from util import empty_or_not, get_parking_spots_bboxes


def calc_diff(im1, im2):
    return np.abs(np.mean(im1) - np.mean(im2))


# File Paths
Video_path = "/home/harish/computer_vision/parking/parking_1920_1080_loop.mp4"
mask_crop_path = "/home/harish/computer_vision/parking/mask_1920_1080.png"

# Load mask image in grayscale
mask = cv2.imread(mask_crop_path, 0)

# Connected components analysis on mask image array
connected_components = cv2.connectedComponentsWithStats(mask, 4, cv2.CV_32S)

# Extract parking spots bboxes using util function
spots = get_parking_spots_bboxes(connected_components)
print("Total spots detected:", len(spots))

# Open Video Capture
cap = cv2.VideoCapture(Video_path)

# Initialize status with 0 (or False) instead of None so sum() doesn't fail
spots_status = [0 for _ in spots]
diffs = [0.0 for _ in spots]
previous_frame = None
step = 30
frame_nmr = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_nmr % step == 0 and previous_frame is not None:
        for spot_indx, spot in enumerate(spots):
            x1, y1, w, h = spot
            spot_crop = frame[y1 : y1 + h, x1 : x1 + w, :]
            diffs[spot_indx] = calc_diff(
                spot_crop, previous_frame[y1 : y1 + h, x1 : x1 + w, :]
            )

    # Har 'step' frames baad ML model run karein (for optimization)
    if frame_nmr % step == 0:
        # ✅ FIX: Use 'is None' instead of '== None'
        if previous_frame is None:
            arr_ = range(len(spots))
        else:
            max_d = np.amax(diffs) if np.amax(diffs) > 0 else 1.0
            arr_ = [j for j in np.argsort(diffs) if (diffs[j] / max_d) > 0.4]

        for spot_indx in arr_:
            x1, y1, w, h = spots[spot_indx]
            spot_crop = frame[y1 : y1 + h, x1 : x1 + w, :]

            # Individual status nikaal kar list me save karein
            st = empty_or_not(spot_crop)
            spots_status[spot_indx] = 1 if st else 0

        previous_frame = frame.copy()

    # Parking bounding boxes draw karein
    for spot_indx, spot in enumerate(spots):
        x1, y1, w, h = spot
        status = spots_status[spot_indx]

        if status:
            # Green rectangle if spot is EMPTY
            cv2.rectangle(frame, (x1, y1), (x1 + w, y1 + h), (0, 255, 0), 2)
        else:
            # Red rectangle if spot is NOT EMPTY
            cv2.rectangle(frame, (x1, y1), (x1 + w, y1 + h), (0, 0, 255), 2)

    # Available spots count text overlay
    cv2.putText(
        frame,
        "Available Spots: {}/{}".format(sum(spots_status), len(spots_status)),
        (100, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2,
    )

    cv2.imshow("Parking Spot Detection", frame)

    if cv2.waitKey(25) & 0xFF == ord("q"):
        break

    frame_nmr += 1

cap.release()
cv2.destroyAllWindows()

/home/harish/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/lib/python3/dist-packages/ipykernel

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/lib/python3/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/lib/python3/dist-packages/traitlets/config/application.py", line 982, in launch_instance
    app.start()
  File "/usr/lib/python3/dist-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/usr/lib/python3/dist-packages/tornado/platform/asyncio.py", line 205, in s

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



/home/harish/.local/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Total spots detected: 396


QFontDatabase: Cannot find font directory /home/harish/.local/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/harish/.local/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/harish/.local/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/harish/.local/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/harish/.local/lib/python3.12/site-pa